# Notebook 05 — Modèle Elastic Net (affichage des résultats)

> ⚠️ **Ce notebook ne calcule plus rien.** Depuis la réorganisation du projet, tout le
> calcul est fait par le script `scripts/etape05_modele_elastic_net.py`, à lancer **à la main**, depuis la racine du
> projet :
>
> ```bash
> python scripts/etape05_modele_elastic_net.py
> ```
>
> Ce notebook se contente de **lire et afficher** ce que ce script a produit : les gros
> fichiers de sortie (chemins dans `config.py`) et le rapport d'exécution `05_elastic_net`
> (`outputs/rapports/`, voir `rapports.py`), qui contient tous les compteurs et petits
> tableaux de diagnostic autrefois imprimés au fil des cellules.
>
> Il est donc **léger et ré-exécutable à volonté** (`Run All` en quelques secondes), sans
> jamais relancer un calcul. Si tu changes un paramètre dans `config.py`, relance d'abord
> le script, puis ce notebook.


**Ce que le script a fait :**
1. Construire les mêmes **fenêtres** qu'au notebook 04
2. Pour **chaque fenêtre**, chercher les meilleurs hyperparamètres `(alpha, l1_ratio)` sur
   la **validation de cette fenêtre** (jamais de validation croisée aléatoire), puis
   ré-entraîner et évaluer une seule fois sur son test
3. Mettre bout à bout les prédictions et calculer le **R²_oos à la GKX**
4. Mesurer la **stabilité de la sélection de variables** sur toutes les fenêtres (section 6bis)
5. Sauvegarder modèle, prédictions, résultats, et ajouter une ligne au journal

## 0. Import et chargement des résultats

In [ ]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import config
import rapports

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)

rap = rapports.charger('05_elastic_net')
print(rap.resume())

resultats_finaux = pd.read_parquet(config.FICHIER_RESULTATS_ELASTIC_NET)
resultats_par_fenetre = pd.read_parquet(config.FICHIER_RESULTATS_ELASTIC_NET_PAR_FENETRE)

## 1. Données et fenêtres d'entraînement

In [ ]:
print(f"Panel complet : {tuple(rap.valeur('shape_panel'))}")
print(f"Periode disponible dans le panel : {rap.valeur('periode_panel')[0]} a {rap.valeur('periode_panel')[1]}")
print(f"Debut d'entrainement impose (config.ANNEE_DEBUT_ENTRAINEMENT) : {rap.valeur('annee_debut_entrainement')}")
print(f"Panel effectivement utilise : {tuple(rap.valeur('shape_panel_entrainement'))} "
      f"({rap.valeur('periode_entrainement')[0]} a {rap.valeur('periode_entrainement')[1]})")
print(f"Mode : {rap.valeur('type_fenetre')} | {rap.valeur('n_fenetres')} fenetres generees")
print(f"Predicteurs : {rap.valeur('n_predicteurs')}")
rap.table('resume_fenetres')

## 2. Le R² hors-échantillon à la Gu, Kelly & Xiu (2020)

Même métrique que les notebooks 04 et 06, indispensable pour rester comparable :

`R²_oos = 1 - Σ(y - ŷ)² / Σ(y²)`

## 3. Recherche d'hyperparamètres, fenêtre par fenêtre

L'Elastic Net a 2 hyperparamètres : `alpha` (force globale de la régularisation) et
`l1_ratio` (équilibre entre Lasso et Ridge). Pour **chaque fenêtre**, le script teste toute
la grille de `config.py`, chaque combinaison entraînée sur le `train` de cette fenêtre et
évaluée sur sa `validation` — **jamais** de validation croisée aléatoire (`cross_val_score`
classique), qui mélangerait des périodes temporelles différentes et créerait une fuite.

⚠️ Le `test` de chaque fenêtre ne sert **qu'une seule fois**, à la toute fin, une fois les
hyperparamètres déjà choisis sur la `validation` de cette même fenêtre.

In [ ]:
print("Parametres SPECIFIQUES utilises (config.py) :")
for cle, valeur in rap.valeur('params_specifiques').items():
    print(f"  {cle} = {valeur}")
print()
print(f"Duree totale (recherche d'hyperparametres incluse) : "
      f"{rap.valeur('duree_entrainement_secondes'):.1f} s")
print()
resultats_par_fenetre

## 3bis. La grille complète : tous les modèles testés au dernier lancement

La section précédente ne montre que la combinaison **gagnante** de chaque fenêtre. Or le
script en a entraîné bien plus : `alpha` × `l1_ratio` combinaisons, **pour chaque fenêtre**.
Ne regarder que le gagnant laisse de côté l'information la plus utile — la **forme** de la
surface de performance.

Un optimum *plat* (plusieurs combinaisons voisines à performances quasi identiques) est un
bon signe : le choix est robuste, une légère erreur de réglage ne coûte presque rien. Un
optimum *pointu* (un seul point nettement au-dessus) doit alerter : il peut être un artefact
du bruit de cette période de validation précise.

ℹ️ Les tableaux ci-dessous concernent **uniquement le dernier lancement** du script (celui
qui a produit le rapport chargé en section 0). Pour comparer plusieurs lancements avec des
paramètres différents, c'est le notebook 08 qui s'en charge.

In [ ]:
grille = rap.table('grille_complete')

print(f"{rap.valeur('n_combinaisons_grille')} combinaisons x {rap.valeur('n_fenetres')} fenetres "
      f"= {rap.valeur('n_modeles_entraines_grille')} modeles entraines au total.")
print()
print("Combinaison retenue par fenetre :")
display(grille[grille['selectionnee']][
    ['fenetre', 'annee_test', 'alpha', 'l1_ratio',
     'r2_oos_train', 'r2_oos_validation', 'ecart_train_validation']
].round(4).set_index('fenetre'))

def _etiquettes(valeurs):
    # 1e-05 plutot que 9.999999999999e-06 : les flottants d'une grille se lisent mieux en %g
    return [f"{v:g}" if isinstance(v, float) else str(v) for v in valeurs]


def tracer_heatmap(pivot, titre, etiquette_couleur, cmap, format_valeur="{:.4f}"):
    # Heatmap annotee a partir d'un tableau croise (2 hyperparametres en lignes/colonnes)
    valeurs = pivot.values.astype(float)
    fig, ax = plt.subplots(figsize=(1.7 * len(pivot.columns) + 3.5,
                                    0.75 * len(pivot.index) + 2.5))
    image = ax.imshow(valeurs, cmap=cmap, aspect='auto')

    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(_etiquettes(pivot.columns))
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(_etiquettes(pivot.index))
    ax.set_xlabel(pivot.columns.name)
    ax.set_ylabel(pivot.index.name)
    ax.set_title(titre)

    milieu = (np.nanmax(valeurs) + np.nanmin(valeurs)) / 2
    for i in range(valeurs.shape[0]):
        for j in range(valeurs.shape[1]):
            v = valeurs[i, j]
            if np.isnan(v):
                continue
            ax.text(j, i, format_valeur.format(v), ha='center', va='center',
                    fontsize=9, color='white' if v < milieu else 'black')

    fig.colorbar(image, ax=ax, label=etiquette_couleur)
    plt.tight_layout()
    plt.show()


def tracer_heatmaps_par_fenetre(grille, lignes, colonnes, valeur, titre,
                                etiquette_couleur, cmap, format_valeur="{:.4f}"):
    # Une heatmap par fenetre, cote a cote, sur une MEME echelle de couleurs
    # (indispensable pour comparer les fenetres entre elles d'un coup d'oeil).
    # Le cadre rouge marque la combinaison reellement retenue par cette fenetre.
    numeros = sorted(grille['fenetre'].unique())
    pivots = {n: grille[grille['fenetre'] == n].pivot_table(
                  index=lignes, columns=colonnes, values=valeur, aggfunc='mean')
              for n in numeros}

    toutes_valeurs = np.concatenate([p.values.astype(float).ravel() for p in pivots.values()])
    mini, maxi = np.nanmin(toutes_valeurs), np.nanmax(toutes_valeurs)
    milieu = (mini + maxi) / 2

    premier = pivots[numeros[0]]
    fig, axes = plt.subplots(
        1, len(numeros), squeeze=False,
        figsize=(1.9 * len(premier.columns) * len(numeros) + 2.5,
                 0.75 * len(premier.index) + 3))

    for ax, n in zip(axes[0], numeros):
        pivot = pivots[n]
        valeurs = pivot.values.astype(float)
        image = ax.imshow(valeurs, cmap=cmap, aspect='auto', vmin=mini, vmax=maxi)

        ax.set_xticks(range(len(pivot.columns)))
        ax.set_xticklabels(_etiquettes(pivot.columns))
        ax.set_yticks(range(len(pivot.index)))
        ax.set_xlabel(pivot.columns.name)
        if n == numeros[0]:
            ax.set_yticklabels(_etiquettes(pivot.index))
            ax.set_ylabel(pivot.index.name)
        else:
            # etiquettes Y sur le premier graphique seulement : sinon elles debordent
            # sur le graphique voisin (l'echelle est commune, donc elles sont identiques)
            ax.set_yticklabels([])

        annee = grille.loc[grille['fenetre'] == n, 'annee_test'].iloc[0]
        ax.set_title(f"Fenetre {n} (test {annee})", fontsize=11)

        for i in range(valeurs.shape[0]):
            for j in range(valeurs.shape[1]):
                v = valeurs[i, j]
                if np.isnan(v):
                    continue
                ax.text(j, i, format_valeur.format(v), ha='center', va='center',
                        fontsize=8, color='white' if v < milieu else 'black')

        gagnante = grille[(grille['fenetre'] == n) & grille['selectionnee']]
        if len(gagnante):
            ligne_gagnante = gagnante.iloc[0]
            index_liste, colonnes_liste = list(pivot.index), list(pivot.columns)
            if ligne_gagnante[lignes] in index_liste and ligne_gagnante[colonnes] in colonnes_liste:
                i = index_liste.index(ligne_gagnante[lignes])
                j = colonnes_liste.index(ligne_gagnante[colonnes])
                ax.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False,
                                           edgecolor='red', linewidth=2.5))

    fig.suptitle(titre)
    fig.colorbar(image, ax=axes[0].tolist(), label=etiquette_couleur, fraction=0.025)
    plt.show()

### R²_oos de validation, moyenné sur les fenêtres

⚠️ Chaque case est la **moyenne du R²_oos de validation sur toutes les fenêtres** pour cette
combinaison. Une combinaison peut donc y paraître médiocre tout en ayant gagné une fenêtre
particulière — le tableau détaillé plus bas permet de vérifier.

In [ ]:
pivot_validation = grille.pivot_table(index='alpha', columns='l1_ratio',
                                      values='r2_oos_validation', aggfunc='mean')
tracer_heatmap(pivot_validation,
               "Elastic Net -- R2_oos de validation moyen (toutes fenetres)",
               "R2_oos validation", cmap='viridis')

#### La même chose, fenêtre par fenêtre

La heatmap précédente moyenne tout : elle dit quelle combinaison marche bien **en général**,
pas si ce « en général » cache des périodes contradictoires. Les heatmaps ci-dessous
décomposent par fenêtre, sur une **échelle de couleurs commune** pour qu'elles restent
comparables entre elles. Le **cadre rouge** marque la combinaison réellement retenue par
chaque fenêtre — le choix de chaque fenêtre est indépendant de celui des autres.

C'est la vue la plus parlante pour juger la stabilité du réglage :

- **Zones foncées au même endroit sur toutes les fenêtres** → l'optimum est stable dans le
  temps, le choix est robuste.
- **Cadre rouge qui saute d'un coin à l'autre** → l'optimum se déplace d'une période à
  l'autre. Ce n'est pas un défaut, mais ça signifie qu'une combinaison unique figée pour tout
  l'historique serait un mauvais choix — et ça justifie précisément le ré-entraînement par
  fenêtre.

In [ ]:
tracer_heatmaps_par_fenetre(grille, lignes='alpha', colonnes='l1_ratio',
                            valeur='r2_oos_validation',
                            titre="Elastic Net -- R2_oos de validation, fenetre par fenetre (cadre rouge = combinaison retenue)",
                            etiquette_couleur="R2_oos validation", cmap='viridis')

In [ ]:
resume_grille = (grille.groupby(['alpha', 'l1_ratio'])
                 .agg(r2_train=('r2_oos_train', 'mean'),
                      r2_validation=('r2_oos_validation', 'mean'),
                      ecart=('ecart_train_validation', 'mean'),
                      coefs_non_nuls=('n_coefficients_non_nuls', 'mean'),
                      fois_selectionnee=('selectionnee', 'sum'))
                 .sort_values('r2_validation', ascending=False))

print("Toutes les combinaisons, classees par R2_oos de validation moyen :")
resume_grille.round(4)

### L'écart train − validation : quelle combinaison sur-apprend le moins ?

Le R²_oos de validation seul ne dit pas **comment** une combinaison arrive à son score. Deux
modèles peuvent obtenir la même validation avec des comportements opposés : l'un colle au
train et généralise mal, l'autre apprend peu mais transfère bien.

L'écart `R²_train − R²_validation` mesure directement cette différence :

- **Écart large et positif** → le modèle explique bien le passé mais beaucoup moins la
  période suivante : sur-apprentissage. Sur l'Elastic Net, c'est le symptôme d'un `alpha`
  trop faible (régularisation insuffisante).
- **Écart proche de zéro** → le modèle généralise ce qu'il a appris. C'est ce qu'on veut.
- **Écart négatif** (validation meilleure que train) → arrive avec un signal très faible et
  une forte régularisation : le modèle prédit presque zéro partout, et la période de
  validation lui est simplement plus favorable. Ce n'est pas un bon signe en soi, plutôt un
  indicateur que le modèle n'apprend quasiment rien.

⚠️ Cette section est un **diagnostic**, pas le critère utilisé par le script : la sélection
reste faite sur le meilleur R²_oos de validation. Si tu décides de sélectionner sur l'écart
(ou sur un compromis entre les deux), il faut changer le critère dans
`scripts/etape05_modele_elastic_net.py` — la ligne `index_meilleur = ...`.

In [ ]:
pivot_ecart = grille.pivot_table(index='alpha', columns='l1_ratio',
                                 values='ecart_train_validation', aggfunc='mean')
tracer_heatmap(pivot_ecart,
               "Elastic Net -- ecart R2 train moins validation (toutes fenetres)\n"
               "proche de 0 = generalise bien, eleve = sur-apprentissage",
               "ecart train - validation", cmap='RdYlGn_r')

#### L'écart, fenêtre par fenêtre

Même décomposition pour l'écart train − validation. À surveiller ici : une combinaison dont
l'écart se creuse au fil des fenêtres sur-apprend de plus en plus à mesure que le train
grandit (en mode `expanding`) — un signe qu'il faudrait renforcer la régularisation sur les
fenêtres tardives.

In [ ]:
tracer_heatmaps_par_fenetre(grille, lignes='alpha', colonnes='l1_ratio',
                            valeur='ecart_train_validation',
                            titre="Elastic Net -- ecart R2 train moins validation, fenetre par fenetre",
                            etiquette_couleur="ecart train - validation", cmap='RdYlGn_r')

In [ ]:
print("Les memes combinaisons, classees cette fois par ecart croissant :")
print("(la colonne r2_validation reste affichee : un ecart faible sur un modele qui")
print(" ne predit rien n'a aucun interet -- les deux criteres se lisent ensemble)")
resume_grille.sort_values('ecart').round(4)

## 4. Évaluation hors-échantillon pooled (toutes les fenêtres bout à bout)

⚠️ Comme au notebook 04 : on met toutes les prédictions bout à bout **d'abord**, puis on
calcule un seul R²_oos — pas une moyenne des R² par fenêtre, qui donnerait un poids
disproportionné aux fenêtres les plus courtes.

In [ ]:
ligne = resultats_finaux.iloc[0]
print(f"R2_oos train      (pooled, toutes fenetres) : {ligne['r2_oos_train']:.4f}")
print(f"R2_oos validation (pooled, toutes fenetres) : {ligne['r2_oos_validation']:.4f}")
print(f"R2_oos test       (pooled, toutes fenetres) : {ligne['r2_oos_test']:.4f}")
print()
resultats_finaux

## 5. Évolution du R²_oos et des hyperparamètres au fil des fenêtres

ℹ️ Si `alpha` varie beaucoup d'une fenêtre à l'autre, c'est un signe que la meilleure force
de régularisation dépend fortement de la période — pas forcément un problème, mais un
diagnostic qu'un unique jeu d'hyperparamètres pour tout le projet ne pourrait pas montrer.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(resultats_par_fenetre['annee_test'], resultats_par_fenetre['r2_oos_test'],
             marker='o', color='tab:blue')
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].set_xlabel("Annee(s) de test de la fenetre")
axes[0].set_ylabel("R2_oos test")
axes[0].set_title("Elastic Net : R2_oos test par fenetre")
axes[0].tick_params(axis='x', rotation=45)

axes[1].plot(resultats_par_fenetre['annee_test'], resultats_par_fenetre['alpha'],
             marker='o', color='tab:orange')
axes[1].set_yscale('log')
axes[1].set_xlabel("Annee(s) de test de la fenetre")
axes[1].set_ylabel("alpha choisi (echelle log)")
axes[1].set_title("Elastic Net : force de regularisation choisie par fenetre")
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 6. Coup d'œil rapide aux coefficients (modèle de la dernière fenêtre)

L'Elastic Net ramène certains coefficients exactement à 0 (sélection automatique de
variables). On regarde ici le modèle de la **dernière fenêtre** — un simple instantané. La
section 6bis agrège la sélection sur **toutes** les fenêtres pour distinguer les variables
régulièrement retenues de celles sélectionnées par accident sur une seule période.

In [ ]:
coefficients = rap.table('coefficients_derniere_fenetre')['coefficient']

print(f"Nombre de variables ramenees exactement a 0 : {rap.valeur('n_coefficients_a_zero')} / "
      f"{rap.valeur('n_predicteurs')}")
print()
print("Top 15 des variables les plus importantes (en valeur absolue) :")
coefficients.head(15)

## 6bis. Stabilité de la sélection de variables (toutes les fenêtres)

Un coefficient non nul sur la **dernière fenêtre** ne dit rien sur sa **fiabilité** : une
variable peut être retenue une année et ignorée la suivante, simplement parce que la
meilleure combinaison `(alpha, l1_ratio)` a légèrement changé. L'idée de la **sélection de
stabilité** (*stability selection*, Meinshausen & Bühlmann, 2010) est de regarder, sur
**toutes** les fenêtres, à quelle fréquence chaque prédicteur a été gardé — l'équivalent,
pour un modèle régularisé, de la significativité par Fama-MacBeth du notebook 04 (les
coefficients de l'Elastic Net n'ont pas de loi d'échantillonnage classique exploitable pour
un test de significativité au sens habituel).

Pour chaque prédicteur :
- **Fréquence de sélection** : % des fenêtres où son coefficient est non nul.
- **Coefficient moyen (fenêtres sélectionnées)** : sa magnitude moyenne, uniquement sur les
  fenêtres où il a été gardé (une moyenne sur toutes les fenêtres, zéros compris, écraserait
  artificiellement les variables rarement sélectionnées).
- **Signe majoritaire** : cohérence du signe d'une fenêtre à l'autre (un signe qui change
  souvent est un signal d'instabilité, même si la fréquence de sélection est élevée).

In [ ]:
stabilite = pd.read_parquet(config.FICHIER_IMPORTANCE_ELASTIC_NET)

print(f"Coefficients agreges sur {rap.valeur('n_fenetres')} fenetres x "
      f"{rap.valeur('n_predicteurs')} predicteurs.")
print()
print(f"{rap.valeur('stab_n_toujours_selectionnes')} / {rap.valeur('stab_n_predicteurs')} "
      "predicteurs selectionnes dans TOUTES les fenetres.")
print(f"{rap.valeur('stab_n_jamais_selectionnes')} / {rap.valeur('stab_n_predicteurs')} "
      "predicteurs jamais selectionnes (coefficient toujours ramene a 0).")
print()
print("Top 20 des predicteurs les plus stables (frequence de selection, puis magnitude) :")
stabilite.head(20).round(3)

In [ ]:
top20_stab = stabilite.head(20).sort_values('frequence_selection_pct')
couleurs = ['tab:blue' if s > 0 else 'tab:red' for s in top20_stab['signe_dominant']]

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(top20_stab.index, top20_stab['frequence_selection_pct'], color=couleurs)
ax.set_xlim(0, 100)
ax.set_xlabel("Frequence de selection (% des fenetres, coefficient != 0)")
ax.set_title("Top 20 des variables les plus souvent selectionnees par l'Elastic Net\n"
             "(bleu = signe dominant positif, rouge = negatif)")
plt.tight_layout()
plt.show()

In [ ]:
# Detail brut : le coefficient de chaque predicteur, fenetre par fenetre
# (utile pour reperer d'ou vient une instabilite reperee ci-dessus)
rap.table('coefficients_par_fenetre').T.head(20).round(4)

## 7 & 8. Ce que le script a sauvegardé, et le journal des expériences

Ici, les paramètres **spécifiques** sont `grille_alpha`, `grille_l1_ratio` et `max_iter` :
deux lancements avec la même grille mais un `TYPE_FENETRE` différent apparaîtront comme deux
lignes d'un **même** tableau au notebook 08 ; deux grilles différentes donneront deux
tableaux distincts.

In [ ]:
print("Modele final (derniere fenetre) :", config.FICHIER_MODELE_ELASTIC_NET)
print("Predictions (toutes fenetres)   :", config.FICHIER_PREDICTIONS_ELASTIC_NET)
print("Resultats pooled                :", config.FICHIER_RESULTATS_ELASTIC_NET)
print("Resultats par fenetre           :", config.FICHIER_RESULTATS_ELASTIC_NET_PAR_FENETRE)
print("Stabilite de selection          :", config.FICHIER_IMPORTANCE_ELASTIC_NET)
print()
print("Cle de cette experience :", rap.valeur('cle_experience'))
if rap.valeur('experience_ajoutee_au_journal'):
    print("-> ligne AJOUTEE au journal des experiences.")
else:
    print("-> experience deja presente dans le journal (memes parametres), rien ajoute : pas de doublon.")
print()
rap.table('apercu_predictions')

## 9. Résumé

- Hyperparamètres (`alpha`, `l1_ratio`) choisis par recherche sur grille, ré-évalués **à
  chaque fenêtre** sur sa propre `validation` (jamais de validation croisée aléatoire,
  jamais sur `test`).
- Même métrique et même logique de fenêtres que le notebook 04, pour une comparaison équitable.
- **Stabilité de la sélection (section 6bis)** : fréquence de sélection, magnitude moyenne
  et cohérence de signe, agrégées sur **toutes** les fenêtres.

**Étape suivante :** `python scripts/etape06_modele_lightgbm.py`.